<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Camera Calibration</b></h1>
</div>

## Requirements and Approach

This notebook mirrors the exact **13-task execution order** used by `camera_calibration.ipynb`. Each section states the engineering requirement, selected method and acceptance condition for the corresponding implementation task.

## Global Requirements

| Requirement | Value |
| --- | --- |
| Input | Sorted JPEG calibration images |
| Internal corners | $8 \times 6$ |
| Square size | $0.03\,\mathrm{m}$ |
| Minimum valid views | 3 |
| Homography solver | Normalized DLT + SVD |
| Intrinsic solver | Zhang constraints + SVD |
| Pose model | Pinhole camera, no distortion |
| Evaluation | Reprojection error in pixels |

## 1. Load the Sorted JPEG Calibration Images

**Approach:** use `Path("../data/calibration_images")` and `sorted(...glob("*.jpg"))`.

**Acceptance:** directory exists, at least one JPEG is found, and processing order is deterministic.

## 2. Detect and Refine Chessboard Corners

**Approach:** `cv2.findChessboardCorners` followed by `cv2.cornerSubPix` with an $11\times11$ window, maximum 30 iterations and epsilon $0.001$.

**Acceptance:** each retained view returns exactly 48 finite image points.

## 3. Build the Planar World Coordinates

**Approach:** generate a regular $8\times6$ planar grid with $0.03$ m spacing.

**Acceptance:** every retained view has 48 planar points with the same ordering as its detected image points.

## 4. Compute $T_{\mathrm{image}}$ and $T_{\mathrm{plane}}$

**Approach:** compute a similarity transform from centroid and mean radius for both point sets.

**Acceptance:** normalized points are centred at the origin with mean distance $\sqrt{2}$.

## 5. Build the DLT Matrix $Q$ and Solve $Q\mathbf{h}=0$ by SVD

**Approach:** build a $2N\times9$ matrix $Q$ from the normalized correspondences and solve the homogeneous system by SVD.

**Acceptance:** reshape the last right singular vector into the normalized $3\times3$ homography $H_n$.

## 6. Denormalize Each Homography

**Approach:** denormalize with

$$
H=T_{\mathrm{image}}^{-1}H_nT_{\mathrm{plane}}
$$

then divide by $H_{33}$.

**Acceptance:** $H$ contains only finite values and its scale is fixed with $H_{33}=1$.

## 7. Build the Zhang Matrix $V$ and Solve $Vb=0$ by SVD

**Approach:** compute $v_{12}$ and $v_{11}-v_{22}$ from every homography, stack all constraints into $V$, then solve by SVD.

**Acceptance:** obtain a finite six-element homogeneous vector $b$ and report $\lVert Vb\rVert$.

## 8. Recover $\alpha,\beta,\gamma,u_0,v_0$ and Construct $K$

**Approach:** apply Zhang's closed-form intrinsic recovery using $b$, including homogeneous sign correction when needed.

**Acceptance:** produce finite $\alpha,\beta,\gamma,u_0,v_0$ and a finite $3\times3$ intrinsic matrix $K$.

## 9. Recover $R$ and $t$ for Every Retained View

**Approach:** decompose $K^{-1}H$ to obtain the first two rotation columns and translation, construct the third rotation column by cross product, then project the approximate rotation to the nearest proper rotation by SVD.

**Acceptance:** $R^TR\approx I$ and $\det(R)\approx1$.

## 10. Reproject the $Z=0$ Calibration Points

**Approach:** convert the planar points to $[X,Y,0]^T$, transform them with $(R,t)$, multiply by $K$ and dehomogenize.

**Acceptance:** one predicted pixel coordinate is obtained for every detected corner.

## 11. Compute Point-wise Errors, Mean Error and RMSE

**Approach:** compute point-wise Euclidean residual magnitudes.

**Acceptance:** provide per-view mean error/RMSE and overall mean error/RMSE, all finite and expressed in pixels.

## 12. Produce and Save the Six Required Diagnostic Figures

**Approach:** generate and save the six required visual diagnostics.

**Acceptance:** all six PNG files exist in `../outputs/figures/`.

### 12.1 Homography Estimation Pipeline
### 12.2 Mean Reprojection Error by View
### 12.3 Reprojection Error Distribution
### 12.4 Detected Chessboard Corners
### 12.5 Estimated Camera Poses
### 12.6 Detected vs Reprojected Points

## 13. Run the Numerical and Output-file Validation Checks

**Approach:** perform final structural, numerical and filesystem checks.

**Acceptance:** valid $K$, valid rotations, finite errors, and all six figures present. Any failure must raise an explicit error rather than silently pass.

## Requirement-to-Code Traceability

| Task | Main implementation location |
| ---: | --- |
| 1 | sorted JPEG loading cell |
| 2 | `detect_and_refine_corners()` + Task 2 processing loop |
| 3 | `build_planar_points()` |
| 4 | `homogenize()` + `normalize_trans()` |
| 5 | `build_dlt_matrix()` + SVD of $Q$ |
| 6 | `denormalize_homography()` |
| 7 | `v_ij()` + `zhang_constraints()` + SVD of $V$ |
| 8 | intrinsic-recovery cell |
| 9 | `recover_extrinsic(K, H)` |
| 10 | reprojection cell |
| 11 | residual / mean / RMSE cells |
| 12 | six figure-generation cells |
| 13 | final validation cell |

Every numbered task in the implementation notebook now has its executable code immediately below the matching heading.